## Classification and Logistic Regression

*Note: The code in this notebook was generated with the assistance of Claude.*

I will be adding to this notebook throughout the week! 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# This dataset can be used to predict whether a patient has a 10-year risk of future CHD (coronary heart disease)
fram = pd.read_csv("../data/framingham.csv")
fram.head()

In [ ]:
# Do we have any missing data? 
fram.isna().sum()

In [ ]:
# For simplicity, we'll drop rows with any missing values (about 14% of rows)
# We will come back to this dataset to look more at imputation
# For now, we are going to just use rows that have all the data
fram = fram.dropna()
fram.shape

In [ ]:
# What fraction of people developed coronary heart disease (CHD) within 10 years?
fram['TenYearCHD'].value_counts(normalize=True)

In [ ]:
# We want to predict whether or not someone develops CHD within 10 years (1 or 0) based on a variety of risk factors
y = fram['TenYearCHD']

# There are three different types of variables in this dataset:
# - education is categorical (levels 1-4), so we one-hot encode it
# - binary variables are already 0/1, so we leave them alone
# - numeric variables
cats = ["education"]
bins = ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes"]
nums = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]

# Note: For this first passthrough, we going to use
# All of the numeric veriables
# Only a single categorical variable (Male or female)
male = ["male"]
X = fram[male + nums]

In [ ]:
# Let's build a logistic regression model
# Note: We are NOT going to scale anything for this first go around, to demonstrate why scaling is important
# This is problematic and in practice, we would scale everything first. 

logreg = LogisticRegression().fit(X,y)

In [ ]:
# We'll come back to the warning later. 
# Let's look at predicted vs actual CHD, and look at the confusion matrix

p = logreg.predict_proba(X)
p

In [ ]:
# Note that this returns probability estimates for 0 or 1, and if we add each pair it should sum to 1. 
# We can look at just the second column (prediction of CHD) and get information about both

p = logreg.predict_proba(X)[:, 1] # second column
y_hat = (p >=0.5).astype(int) # if p >= 0.5, set to 1 (we can vary this threshold!)

In [ ]:
# Now let's create a dataframe of actual vs predicted
results = pd.DataFrame(
    {
    "Actual_CHD": y,
    "Predicted_prob_CHD": p.round(3),
    "Predicted_label": y_hat
})

results

In [ ]:
# Now we look at the confusion matrix, which gives us the following information:
# [[True Negative, False Positive],[False Negative, True Positive]]

confusion_matrix(y, y_hat)

In [ ]:
# We can put this in a slightly more digestible form
tn, fp, fn, tp = confusion_matrix(y, y_hat).ravel().tolist()
tn, fp, fn, tp

In [ ]:
#We can also normalize the confusion matrix:
confusion_matrix(y, y_hat, normalize="true")


In [ ]:
# We can also plot the confusion matrix to visualize this. 
cm = confusion_matrix(y, y_hat, labels=logreg.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
# Only ~15% of people develop CHD, so very few predicted probabilities exceed 0.5
# What happens to our confusion matrix if we lower the threshold to 0.2?
y_hat_alt = (p >=0.2).astype(int)
confusion_matrix(y, y_hat_alt)

In [ ]:
# We can also normalize
confusion_matrix(y, y_hat_alt, normalize="true")

In [ ]:
# What does changing the threshold tell us about the tradeoff between false negatives and false positives? 
cm = confusion_matrix(y, y_hat_alt, labels=logreg.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
# Now let's go back to the warning we saw earlier and look at the scale of the numeric variables.
fram.describe()
